Our goal is to teach a computer to read text and predict which category that text belongs to.


For example:

    "NASA launched a new satellite" → Space / Science

    "The football team won the match" → Sports

    "Python is a programming language" → Computers

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [ ]:
categories = [
    "comp.graphics",
    "sci.space",
    "rec.sport.baseball",
    "talk.politics.misc"
]

data = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes")
)

X = data.data
y = data.target

print("Total documents:", len(X))
print("Categories:", data.target_names)
print("First document:", X[0][:500])

In [ ]:
#  Check the number of documents in each category
unique, counts = np.unique(y, return_counts=True)

for category_id, count in zip(unique, counts):
    print(data.target_names[category_id], count)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
# Step 4 — Convert text into numerical features

vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training matrix:", X_train_tfidf.shape)
print("Testing matrix:", X_test_tfidf.shape)

In [ ]:
# Step 5 — Train Logistic Regression
model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train_tfidf, y_train)

In [ ]:
# Step 6 — Make predictions

predictions = model.predict(X_test_tfidf)

print(predictions[:10])
print(y_test[:10])

In [ ]:
accuracy = accuracy_score(
    y_test,
    predictions
)

print("Accuracy:", accuracy)

print(
    classification_report(
        y_test,
        predictions,
        target_names=data.target_names
    )
)


In [ ]:
# Step 8 — Inspect the confusion matrix

cm = confusion_matrix(
    y_test,
    predictions
)

print(cm)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=data.target_names,
    yticklabels=data.target_names
)

plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
import numpy as np

# Find documents where prediction is incorrect
wrong_indices = np.where(y_test != predictions)[0]

print("Total incorrect predictions:", len(wrong_indices))

# Inspect the first 5 incorrect predictions
for i in wrong_indices[:5]:

    print("\nActual category:",
          data.target_names[y_test[i]])

    print("Predicted category:",
          data.target_names[predictions[i]])

    print("Document:")
    print(X_test[i][:500])

    print("-" * 80)

In [ ]:
# Select one incorrectly classified document
i = wrong_indices[0]

# Get probability scores for each category
probabilities = model.predict_proba(
    X_test_tfidf[i]
)[0]

# Print the document
print("Document:")
print(X_test[i][:500])

# Print actual category
print("\nActual:")
print(data.target_names[y_test[i]])

# Print predicted category
print("\nPredicted:")
print(data.target_names[predictions[i]])

# Print probabilities
print("\nCategory probabilities:")

for category, probability in zip(
    data.target_names,
    probabilities
):
    print(category, round(probability * 100, 2), "%")

In [ ]:
# Experiment B — Compare algorithms : Train a second model

nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

nb_predictions = nb_model.predict(X_test_tfidf)

print(
    "Naive Bayes Accuracy:",
    accuracy_score(y_test, nb_predictions)
)

print("Logistic Regression:",
      accuracy_score(y_test, predictions))

print("Naive Bayes:",
      accuracy_score(y_test, nb_predictions))

print("NAIVE BAYES REPORT")

print(classification_report(
    y_test,
    nb_predictions
))

In [ ]:
print("Training samples:", X_train_tfidf.shape[0])
print("Testing samples:", X_test_tfidf.shape[0])

print("Training labels:", len(y_train))
print("Testing labels:", len(y_test))
print("TF-IDF features:", X_train_tfidf.shape[1])

In [ ]:
# Investigate the documents your Naive Bayes model classified incorrectly.
wrong_indices = np.where(
    nb_predictions != y_test
)[0]

print("Incorrect predictions:", len(wrong_indices))

for i in wrong_indices[:10]:

    print("\nActual:",
          data.target_names[y_test.iloc[i]]
          if hasattr(y_test, "iloc") else y_test[i])

    print("Predicted:",
          data.target_names[nb_predictions[i]])

In [46]:
# Experiment C — Custom predictions

custom_texts = [
    "NASA launched a spacecraft to explore Mars",
    "The baseball team won the championship",
    "New computer graphics technology is amazing",
    "The government announced a new political policy"
]

custom_tfidf = vectorizer.transform(custom_texts)

custom_predictions = model.predict(custom_tfidf)

for text, prediction in zip(
    custom_texts,
    custom_predictions
):
    print(
        "Text:", text, 
        "| Prediction:", data.target_names[prediction]
    )

Text: NASA launched a spacecraft to explore Mars | Prediction: sci.space
Text: The baseball team won the championship | Prediction: rec.sport.baseball
Text: New computer graphics technology is amazing | Prediction: comp.graphics
Text: The government announced a new political policy | Prediction: talk.politics.misc
